<a href="https://colab.research.google.com/github/gangababupokanati/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [7]:
# Load sample résumés from the lab kit
with open('sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.12 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal (3-week internship at startup)",
    "Solved 250+ DSA problems on LeetCode (Top 5% in college)",
    "Final-year project: ML model for crop yield prediction (Random Forest, 87% accuracy)"
  ],
  "experience_years": 0.25
}


In [8]:
# Empty string — should fail gracefully, not crash
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"Master of Science in Computer Science","institution":"University of Tech","year":2020},{"degree":"Bachelor of Science in Software Engineering","institution":"State College","year":2018}],"skills":["Python","Java","React","SQL","Cloud Computing"],"projects":["E-commerce Platform","Data Analysis Tool"],"experience_years":5.5}


## Day 2 Lab 2B — Errors handled

1. **Markdown fence wrapping** (`\`\`\`json ... \`\`\``)
   The retry prompt asks Gemini to output raw JSON without fences. Triggers on ~5-10% of calls.

2. **Hallucinated phone number when source has none**
   `Optional[str] = None` in Pydantic — model returns `null`, schema validates.

3. **Empty / whitespace-only input**
   Pydantic raises ValidationError with "Field required". Caller catches.

## Sample résumés processed: 3 / 3 successful

In [14]:
from pydantic import BaseModel
from typing import List

class Option(BaseModel):
    label: str        # "A", "B", "C", or "D"
    text: str         # The option text

class MCQQuestion(BaseModel):
    question_number: int
    question: str
    options: List[Option]          # Always 4 options: A, B, C, D
    correct_answer: str            # "A", "B", "C", or "D"
    correct_answer_text: str       # Full text of the correct option

class MCQSet(BaseModel):
    questions: List[MCQQuestion]

In [15]:
import os
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_mcq(raw_text: str, question_number: int, max_retries: int = 1) -> MCQQuestion:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=(
                    f'Extract a single MCQ from this text and return ONLY JSON, no markdown.\n'
                    f'Assign question_number = {question_number}.\n\n{raw_text}'
                ),
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': MCQQuestion.model_json_schema(),
                },
            )
            return MCQQuestion.model_validate_json(resp.text)

        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Self-healing retry: send broken JSON + errors back to Gemini
            fix_prompt = (
                f'Fix this JSON to match the schema. '
                f'Errors: {e}. '
                f'Original: {resp.text}'
            )
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': MCQQuestion.model_json_schema(),
                },
            )
            return MCQQuestion.model_validate_json(resp.text)

In [16]:
# Load the MCQ text file and split on '---' delimiter
with open('dbms_mcqs.txt') as f:
    raw_questions = [q.strip() for q in f.read().split('---') if q.strip()]

print(f'Loaded {len(raw_questions)} MCQs\n')

results = []
for i, raw_q in enumerate(raw_questions):
    try:
        parsed = extract_mcq(raw_q, question_number=i + 1)
        results.append(parsed)
        print(f'Q{parsed.question_number}: {parsed.question[:60]}...')
        for opt in parsed.options:
            marker = " ✓" if opt.label == parsed.correct_answer else ""
            print(f'   {opt.label}) {opt.text}{marker}')
        print(f'   → Correct: {parsed.correct_answer}) {parsed.correct_answer_text}\n')
    except Exception as e:
        print(f'Q{i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}\n')

# Print full structured JSON for the first question
if results:
    print('\n=== Full JSON of first question ===')
    print(results[0].model_dump_json(indent=2))

# Optionally, dump all questions as a single MCQSet JSON
mcq_set = MCQSet(questions=results)
with open('dbms_mcqs_parsed.json', 'w') as f:
    f.write(mcq_set.model_dump_json(indent=2))
print(f'\nAll {len(results)} parsed questions saved to dbms_mcqs_parsed.json')

Loaded 10 MCQs

Q1: What does DBMS stand for?...
   A) Data Backup Management System
   B) Database Management System ✓
   C) Data Based Monitoring System
   D) Distributed Base Management System
   → Correct: B) Database Management System

Q2: Which of the following is NOT a type of database model?...
   A) Relational Model
   B) Hierarchical Model
   C) Sequential Model ✓
   D) Network Model
   → Correct: C) Sequential Model

Q3: What is a primary key in a relational database?...
   A) A key that can have NULL values
   B) A key that uniquely identifies each row in a table ✓
   C) A key used only for indexing
   D) A foreign key in another table
   → Correct: B) A key that uniquely identifies each row in a table

Q4: Which SQL command is used to retrieve data from a database?...
   A) INSERT
   B) UPDATE
   C) SELECT ✓
   D) DELETE
   → Correct: C) SELECT

Q5: What is normalization in DBMS?...
   A) The process of converting data to binary format
   B) The process of organizing data 